# AXUM — Ge'ez OCR training (CNN + BiLSTM + CTC)

Train the **honest HHD-only baseline** for the artefact inscription OCR model.

> **Why HHD-only?** Directly merging HHD-Ethiopic with YaredOffice previously collapsed accuracy
> from ~68% to ~17%. The merge conflated two different tasks (word/line images vs. isolated
> glyphs) and used an unverified Yaredoffice folder→Unicode mapping. Until that class map is
> recovered and verified, YaredOffice is only used for encoder *pretraining*
> (`scripts/pretrain_ocr_glyph_encoder.py`), never merged directly into the CTC training set.

**Before this notebook (on your laptop):**
1. `python scripts/export_colab_training_data.py --data-root data/geez_characters --include-code`
   (bundles `train_raw/` **and** the official IID + 18th-century OOD test splits automatically)
2. Upload **both** zips from `exports/`:
   - `geez_characters_colab.zip` (images + CSV + official test splits)
   - `geez_ocr_colab_code.zip` (current training code)

**Check the export is current:** open `geez_characters_colab.manifest.json` on your laptop —
`ocr_pipeline_fix_id` should be `ctc_filter_alignment_metrics_v1` and `ocr_fixes_present_in_repo`
should be `true`.

**Runtime:** GPU recommended (Runtime → Change runtime type → T4). CPU works but is much slower
(roughly 800 minutes/epoch on a CPU-only machine for the full ~57k-sample train set).

**Baseline defaults:** the weighted sampler and stone-texture augmentation are both **off** below —
this run measures the real, unaugmented HHD signal before any domain adaptation.

**After training:** download `geez_ocr.pth` → place in `models/` on the rover laptop. The official
evaluation cell reports honest CER/CharAcc/SeqAcc on both the IID and 18th-century OOD test sets,
matching `scripts/evaluate_ocr.py`.

In [ ]:
# ── 1. GPU check ─────────────────────────────────────────────
import torch

USE_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USE_GPU else "cpu"
print(f"Device: {DEVICE}")
if USE_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Tip: Runtime → Change runtime type → T4 GPU for ~10× faster training")

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────
%%capture
!pip install -q loguru tqdm albumentations opencv-python-headless pillow numpy scipy

In [ ]:
# ── 3. Locate archives or folders (Kaggle or Colab) ────────────
# Kaggle accepts either:
#   A. both ZIP files as notebook inputs, or
#   B. two extracted input folders: geez_characters/ and axum/.
# Kaggle mounts all inputs read-only below /kaggle/input/.
# Colab falls back to its normal interactive file uploader.
from pathlib import Path
import shutil
import sys

IS_KAGGLE = Path("/kaggle/input").exists()
CONTENT = Path("/kaggle/working") if IS_KAGGLE else Path("/content")
AXUM_ROOT = CONTENT / "axum"
DATA_ROOT = CONTENT / "data"
DATA_DIR = DATA_ROOT / "geez_characters"


def find_input(name: str) -> Path | None:
    """Find one Kaggle input by file or directory name, regardless of dataset folder."""
    search_root = Path("/kaggle/input") if IS_KAGGLE else CONTENT
    matches = list(search_root.rglob(name))
    return matches[0] if matches else None


def copy_tree(source: Path, destination: Path) -> None:
    """Copy a read-only Kaggle folder into /kaggle/working only when needed."""
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(source, destination)


if IS_KAGGLE:
    print("Kaggle environment detected.")

    DATA_ZIP = find_input("geez_characters_colab.zip")
    CODE_ZIP = find_input("geez_ocr_colab_code.zip")
    INPUT_DATA_DIR = find_input("geez_characters")
    INPUT_AXUM_ROOT = find_input("axum")

    if DATA_ZIP and CODE_ZIP:
        !unzip -q -o "{DATA_ZIP}" -d "{DATA_ROOT}"
        !unzip -q -o "{CODE_ZIP}" -d "{AXUM_ROOT}"
        archive_mode = "Kaggle ZIP inputs"
    elif INPUT_DATA_DIR and INPUT_AXUM_ROOT:
        copy_tree(INPUT_DATA_DIR, DATA_DIR)
        copy_tree(INPUT_AXUM_ROOT, AXUM_ROOT)
        archive_mode = "Kaggle extracted-folder inputs"
    else:
        visible = sorted(path.name for path in Path("/kaggle/input").iterdir())
        raise FileNotFoundError(
            "Kaggle has no usable AXUM inputs yet. In the right sidebar, click Add Input, "
            "then upload/attach either:\n"
            "  A. geez_characters_colab.zip AND geez_ocr_colab_code.zip, or\n"
            "  B. folders named geez_characters/ AND axum/.\n"
            f"Top-level /kaggle/input entries currently visible: {visible or 'none'}"
        )
else:
    from google.colab import files

    DATA_ZIP = find_input("geez_characters_colab.zip")
    CODE_ZIP = find_input("geez_ocr_colab_code.zip")
    if not DATA_ZIP or not CODE_ZIP:
        print("Upload geez_characters_colab.zip and geez_ocr_colab_code.zip from exports/")
        uploaded = files.upload()
        DATA_ZIP = CONTENT / "geez_characters_colab.zip"
        CODE_ZIP = CONTENT / "geez_ocr_colab_code.zip"

    if not DATA_ZIP.exists() or not CODE_ZIP.exists():
        raise FileNotFoundError(
            "Both archives are required: geez_characters_colab.zip and geez_ocr_colab_code.zip"
        )
    !unzip -q -o "{DATA_ZIP}" -d "{DATA_ROOT}"
    !unzip -q -o "{CODE_ZIP}" -d "{AXUM_ROOT}"
    archive_mode = "Colab upload"

sys.path.insert(0, str(AXUM_ROOT))
print("Archive mode:", archive_mode)
print("AXUM root:", AXUM_ROOT)
print("Data root:", DATA_ROOT)
print("Data directory:", DATA_DIR)

In [ ]:
# ── 4. Verify export has current OCR fixes + CTC-safe labels ─
from src.ocr.pipeline import HHDEthiopicDataset
from src.ocr.training_contract import min_ctc_timesteps, charset_fingerprint
from src.ocr.model import GEEZ_CHARSET
from config import OCR_CTC_SEQ_LEN, OCR_IMG_SIZE

DATA_DIR = DATA_ROOT / "geez_characters"
assert (DATA_DIR / "train_raw" / "image_text_pairs_train.csv").exists(), (
    f"Missing CSV under {DATA_DIR}/train_raw — re-export from laptop"
)

# Code fix markers (post 0%-bug pipeline, current shared training contract)
pipeline_src = (AXUM_ROOT / "src/ocr/pipeline.py").read_text(encoding="utf-8")
contract_present = (AXUM_ROOT / "src/ocr/training_contract.py").exists()
fixes_ok = contract_present and all(
    marker in pipeline_src
    for marker in ("training_contract", "OCR_CTC_SEQ_LEN", "compute_sequence_metrics")
)
print(f"OCR pipeline fixes present: {fixes_ok}")
print(f"Charset: {len(GEEZ_CHARSET)} tokens (fingerprint {charset_fingerprint(GEEZ_CHARSET)})")
print(f"Image size: {tuple(OCR_IMG_SIZE)}  |  CTC sequence budget: {OCR_CTC_SEQ_LEN} timesteps")
assert fixes_ok, "Re-export with --include-code after pulling latest AXUM"

train_ds = HHDEthiopicDataset(str(DATA_DIR), split="train", augment=False, use_stone_augment=False)
val_ds = HHDEthiopicDataset(str(DATA_DIR), split="val", augment=False, use_stone_augment=False)
print(f"Train samples: {len(train_ds):,} | Val: {len(val_ds):,}")

# Spot-check a few labels
for _, text in train_ds.samples[:5]:
    print(f"  len={len(text):2d}  ctc_steps={min_ctc_timesteps(text):2d}  {text[:40]}")

In [ ]:
# ── 5. Training config (interactive) ──────────────────────────
import ipywidgets as widgets
from IPython.display import display

epochs_slider = widgets.IntSlider(value=50, min=5, max=100, step=5, description="Epochs")
batch_slider = widgets.IntSlider(
    value=64 if USE_GPU else 32,
    min=8,
    max=128 if USE_GPU else 64,
    step=8,
    description="Batch",
)
lr_slider = widgets.FloatLogSlider(
    value=0.001,
    base=10,
    min=-4,
    max=-1,
    step=0.1,
    description="LR",
)
weighted_sampler_checkbox = widgets.Checkbox(value=False, description="Weighted sampler")
stone_augment_checkbox = widgets.Checkbox(value=False, description="Stone augment")
seed_input = widgets.IntText(value=42, description="Seed")

display(widgets.VBox([
    epochs_slider, batch_slider, lr_slider,
    weighted_sampler_checkbox, stone_augment_checkbox, seed_input,
]))

EPOCHS = epochs_slider.value
BATCH_SIZE = batch_slider.value
LEARNING_RATE = lr_slider.value
USE_WEIGHTED_SAMPLER = weighted_sampler_checkbox.value
USE_STONE_AUGMENT = stone_augment_checkbox.value
SEED = seed_input.value
SAVE_PATH = AXUM_ROOT / "models" / "geez_ocr.pth"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Epochs={EPOCHS}  batch={BATCH_SIZE}  lr={LEARNING_RATE}  device={DEVICE}  seed={SEED}")
print(f"weighted_sampler={USE_WEIGHTED_SAMPLER}  stone_augment={USE_STONE_AUGMENT}")
print(f"Checkpoint → {SAVE_PATH}")
print("Baseline default: both flags off, to measure the honest unaugmented HHD signal.")
print("Re-run this cell after changing the widgets, then re-run the training cell.")

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────
from scripts.train_ocr import create_weighted_sampler
from src.ocr.pipeline import train_ocr_model

result = train_ocr_model(
    data_dir=str(DATA_DIR),
    save_path=SAVE_PATH,
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    use_weighted_sampler=USE_WEIGHTED_SAMPLER,
    use_stone_augment=USE_STONE_AUGMENT,
    weighted_sampler_fn=create_weighted_sampler,
    device=DEVICE,
    seed=SEED,
)

print("\nBest val loss:", result.get("best_val_loss"))
if result.get("epoch_metrics"):
    last = result["epoch_metrics"][-1]
    print(
        f"Last epoch — CER: {last['cer']:.1%}, "
        f"CharAcc: {last['char_accuracy']:.1%}, "
        f"SeqAcc: {last['sequence_accuracy']:.1%}"
    )

In [ ]:
# ── 7. Quick sanity inference on a val sample ────────────────
import random
from src.ocr.model import load_ocr_model, IDX_TO_CHAR

model = load_ocr_model(SAVE_PATH)
idx = random.randint(0, len(val_ds) - 1)
tensor, label_tensor, _ = val_ds[idx]
true_text = "".join(IDX_TO_CHAR.get(i, "") for i in label_tensor.tolist())

with torch.no_grad():
    log_probs = model(tensor.unsqueeze(0).to(DEVICE))
    pred = model.decode_beam(log_probs)[0]

print("True: ", true_text)
print("Pred: ", pred)
print("Match:", pred == true_text)

## 8. Official evaluation (IID + 18th-century OOD)

The cell below runs the same honest, edit-distance-based evaluation as `scripts/evaluate_ocr.py`
on both official HHD held-out test sets — not the random train/val split used during training.

In [ ]:
# ── 8. Official evaluation (IID + 18th-century OOD) ──────────
import json

from torch.utils.data import DataLoader

from src.ocr.model import IDX_TO_CHAR, load_ocr_model
from src.ocr.pipeline import ctc_collate_fn
from src.ocr.training_contract import compute_sequence_metrics

OFFICIAL_SPLITS = {
    "iid": (
        DATA_DIR / "test/test_rand/image_text_pairs_test_rand.csv",
        DATA_DIR / "test/test_rand/image_rand",
    ),
    "ood-18th": (
        DATA_DIR / "test/test_18th/image_text_pairs_test_18th.csv",
        DATA_DIR / "test/test_18th/image_18th",
    ),
}

eval_model = load_ocr_model(SAVE_PATH).to(DEVICE)
eval_model.eval()

official_results = []
for split_name, (manifest, image_dir) in OFFICIAL_SPLITS.items():
    if not manifest.exists():
        print(f"Skipping {split_name}: {manifest} not found — re-export with official test splits")
        continue
    split_ds = HHDEthiopicDataset(
        str(manifest.parent), split="test", augment=False,
        manifest_path=manifest, image_dir=image_dir,
    )
    loader = DataLoader(split_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=ctc_collate_fn)
    references, predictions = [], []
    with torch.no_grad():
        for images, labels, _input_lengths, label_lengths in loader:
            log_probs = eval_model(images.to(DEVICE))
            texts = eval_model.decode_beam(log_probs)
            offset = 0
            for text, length in zip(texts, label_lengths.tolist()):
                references.append("".join(IDX_TO_CHAR[i] for i in labels[offset:offset + length].tolist()))
                predictions.append(text)
                offset += length
    metrics = compute_sequence_metrics(references, predictions)
    official_results.append({"split": split_name, "samples": len(split_ds), **metrics.to_dict()})
    print(
        f"{split_name}: CER={metrics.cer:.2%}  CharAcc={metrics.character_accuracy:.2%}  "
        f"SeqAcc={metrics.sequence_accuracy:.2%}  n={len(split_ds)}"
    )

eval_log_dir = AXUM_ROOT / "logs"
eval_log_dir.mkdir(parents=True, exist_ok=True)
eval_log_path = eval_log_dir / "ocr_official_evaluation.json"
eval_log_path.write_text(json.dumps(official_results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved: {eval_log_path}")

## Download checkpoint to rover laptop

1. Run the cell below to download `geez_ocr.pth` (and the official evaluation JSON).
2. Copy `geez_ocr.pth` to `models/geez_ocr.pth` in your AXUM project.
3. Re-verify locally any time: `python scripts/evaluate_ocr.py --model models/geez_ocr.pth`

**Metrics guide (real edit-distance based, not positional `zip()`):**
- **CER** — character error rate = (substitutions + deletions + insertions) / reference length; lower is better
- **CharAcc** — `1 - CER`, clipped at 0
- **SeqAcc** — exact full-string match rate (strict; expect low early in training)

In [ ]:
from google.colab import files

assert SAVE_PATH.exists(), "Train first — no checkpoint found"
files.download(str(SAVE_PATH))

eval_json = AXUM_ROOT / "logs" / "ocr_official_evaluation.json"
if eval_json.exists():
    files.download(str(eval_json))